# Chapter 12 — Reproducible Notebooks

**Book alignment:** Debugging AI From First Principles, Chapter 12

**Question this notebook isolates:** The same notebook returns 0.91 / 0.84 / 0.88 across two
machines and one rerun — all green, no error. Treating a notebook run as a pure function of
`(substrate, seed, data, clock)`, can each pin be *attributed* — substrate closes the
cross-machine gap, seeds collapse rerun variance, data locks the rest — one variable class
at a time?

In [ ]:
import numpy as np, random

def notebook_run(*, sklearn="1.4", data="v3", seed=None):
    """A deterministic stand-in for a full training notebook: val_accuracy out."""
    s = seed if seed is not None else random.randint(0, 10**9)   # 'unseeded' = fresh draw each call
    rng = np.random.default_rng(s)
    base = 0.885
    substrate = {"1.2": -0.045, "1.4": 0.0}[sklearn]   # an unpinned package version is an input
    data_off  = {"v2": -0.020, "v3": 0.0}[data]        # data/latest.csv resolves to different bytes
    seed_noise = float(rng.normal(0.0, 0.015))         # unseeded shuffle / init
    return round(base + substrate + data_off + seed_noise, 3)

## 1. Baseline: three green runs, three answers

In [ ]:
author    = notebook_run(sklearn="1.4", data="v3")             # author laptop
colleague = notebook_run(sklearn="1.2", data="v2")             # colleague laptop
author_rerun = notebook_run(sklearn="1.4", data="v3")          # author, same machine, again
print(f"author {author}   colleague {colleague}   author-rerun {author_rerun}")
assert abs(author - colleague) > 0.03                          # a real cross-machine gap
assert author != author_rerun                                 # and it drifts on one machine too
print("all green; none agree -> 'it ran' is not 'it reproduces'")

## 2. Attribute each pin — hold every other class fixed with one shared seed

In [ ]:
# substrate: same seed + data, vary only sklearn -> the seed noise cancels, the offset is bare
sub_effect = notebook_run(sklearn="1.2", data="v3", seed=1) - notebook_run(sklearn="1.4", data="v3", seed=1)
print(f"substrate contribution (sklearn 1.2 vs 1.4): {sub_effect:+.3f}")
assert abs(sub_effect - (-0.045)) < 1e-9

# data: same seed + substrate, vary only the input version
data_effect = notebook_run(sklearn="1.4", data="v2", seed=1) - notebook_run(sklearn="1.4", data="v3", seed=1)
print(f"data contribution (latest.csv v2 vs v3):     {data_effect:+.3f}")
assert abs(data_effect - (-0.020)) < 1e-9

# seeds: pinning the seed makes the run a deterministic function; unpinned it is a fresh draw
assert notebook_run(seed=42) == notebook_run(seed=42)
assert notebook_run(seed=None) != notebook_run(seed=None)
print("seed pinned -> identical; seed unpinned -> every run is a new sample")

## 3. Pin all classes, then confirm with three cold runs

In [ ]:
cold = [notebook_run(sklearn="1.4", data="v3", seed=42) for _ in range(3)]
spread = max(cold) - min(cold)
print("three cold runs, fully pinned:", cold, " spread", round(spread, 6))
assert spread == 0.0
# the pinned number is ONE sample, not an estimate: a trustworthy figure varies many sources
draws = [notebook_run(sklearn="1.4", data="v3", seed=k) for k in range(20)]
print(f"across 20 seeds: mean {np.mean(draws):.3f}  min {min(draws):.3f}  max {max(draws):.3f}")
assert max(draws) - min(draws) > 0.02
print("pin the seed for DEBUGGING (a bisectable function); un-pin it broadly to TRUST the number")

## What we earned

Three green runs, three answers, zero errors — irreproducibility is mostly administrative,
not numerical. Modelling the run as `f(substrate, seed, data, clock)` let each pin be
attributed in isolation: `sklearn 1.2→1.4` was worth −0.045, `latest.csv` −0.020, the
unseeded draw ±0.015. Substrate first (or the seed experiments measure the wrong
distribution), then seeds, then data. The pinned number is one sample; trusting it as an
estimate means varying many sources, not fixing them.

**Notebook 13 / Chapter 13** turns to the input the reproducible run computes on: the data,
before the model.